In [11]:
# If needed:
# !pip install praat-parselmouth praatio pandas numpy

import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import parselmouth
from praatio import tgio

In [12]:
ROOT = Path("/Users/moanason/Downloads/Data_ANA")

# TextGrid tier names – adapt if your tiers have slightly different names
TIER_DIAR_A      = "Diarisation_A"
TIER_DIAR_B      = "Diarisation_B"
TIER_EVENTS      = "TransEvents"
TIER_SYL_A       = "Transcribe_A"
TIER_SYL_B       = "Transcribe_B"

# Condition mapping: c = {0: 'NC1', 1: 'NC2', 2: 'NS'}
COND_CODE_MAP = {
    "NC1": 0,
    "NC2": 1,
    "NS": 2,
}

##### 2 Parse combined TextGrid name → session + condition, Basic utilities: RMS, mean F0, syllable count, overlaps

In [ ]:
def parse_combined_tg_name(name: str):
    """
    Parse combined TextGrid name:
        ref_s##_N(C|S)#_processed.TextGrid
    Returns:
        (session_id:int, cond_str:str, cs_flag:str, rep:str)
        cond_str in {"NC1","NC2","NS"}
    """
    pattern = re.compile(r"ref_s(\d+)_N([CS])(\d*)_processed", re.IGNORECASE)
    m = pattern.search(name)
    if not m:
        return None

    sess_num = int(m.group(1))
    cs_flag = m.group(2).upper()     # C or S
    rep = m.group(3) or "1"

    if cs_flag == "C" and rep == "1":
        cond_str = "NC1"
    elif cs_flag == "C" and rep == "2":
        cond_str = "NC2"
    elif cs_flag == "S":
        cond_str = "NS"
    else:
        raise ValueError(f"Unexpected condition pattern in {name}")

    return sess_num, cond_str, cs_flag, rep


def mono_audio_path(sess_num: int, spk: str, cs_flag: str, rep: str) -> Path:
    sess_str = f"{sess_num:02d}"
    cond_part = f"N{cs_flag}{rep}_processed"
    fname = f"p142_s{sess_str}_{spk}_{cond_part}.wav"
    path = ROOT / fname
    if not path.is_file():
        raise FileNotFoundError(f"Mono audio not found: {path}")
    return path


def compute_rms(samples: np.ndarray) -> float:
    if samples.size == 0:
        return math.nan
    return float(np.sqrt(np.mean(samples**2)))


def mean_f0_in_interval(pitch: parselmouth.Pitch, start: float, end: float) -> float:
    times = pitch.xs()
    freqs = pitch.selected_array["frequency"]
    mask = (times >= start) & (times <= end)
    vals = freqs[mask]
    vals = vals[vals > 0]
    return float(vals.mean()) if vals.size > 0 else math.nan


def rms_in_interval(sound: parselmouth.Sound, sr: float, start: float, end: float) -> float:
    start_samp = int(start * sr)
    end_samp = int(end * sr)
    samples = sound.values[0, start_samp:end_samp]
    return compute_rms(samples)


def count_syllables_in_interval(point_tier: tgio.PointTier, start: float, end: float) -> int:
    count = 0
    for t, label in point_tier.entryList:
        if start <= t <= end:
            count += 1
    return count


def intervals_overlap(a_start, a_end, b_start, b_end) -> float:
    s = max(a_start, b_start)
    e = min(a_end, b_end)
    return max(0.0, e - s)

# event label utilities
def is_backchannel_label(label: str, speaker: int) -> bool:
    if not label:
        return False
    low = label.lower()
    if "backchannel" not in low:
        return False
    if speaker == 0 and ("_a" in low or low.endswith("a")):
        return True
    if speaker == 1 and ("_b" in low or low.endswith("b")):
        return True
    return False


from typing import Optional

def classify_event_label(label: str) -> Optional[str]:
    """
    Map TransEvents label to high-level event type:
    'Gap', 'Silence', 'Overlap', 'Backchannel' or None.
    We only use this for non-Turn events.
    """
    if not label:
        return None
    if not isinstance(label, str):
        return None
        
    low = label.lower()
    if "gap" in low:
        return "Gap"
    if "overlap" in low:
        return "Overlap"
    if "silence" in low or "pause" in low:
        return "Silence"
    if "backchannel" in low:
        return "Backchannel"
    return None


##### Important: define Turn interval excluding backchannel segments

In [ ]:
def diar_interval_is_backchannel(
    events_tier: tgio.IntervalTier,
    start: float,
    end: float,
    speaker: int,
    min_overlap_ratio: float = 0.5,
) -> bool:
    dur = end - start
    if dur <= 0:
        return False

    total_bc_overlap = 0.0
    for ev_start, ev_end, ev_label in events_tier.entryList:
        if not is_backchannel_label(ev_label, speaker):
            continue
        ov = intervals_overlap(start, end, ev_start, ev_end)
        total_bc_overlap += ov

    return (total_bc_overlap / dur) >= min_overlap_ratio


In [ ]:
event_rows = []

tg_paths = sorted(ROOT.glob("ref_s*_N*_processed.TextGrid"))

print(f"Found {len(tg_paths)} combined TextGrids.")

for tg_path in tg_paths:
    parsed = parse_combined_tg_name(tg_path.name)
    if parsed is None:
        print("Skipping (name pattern mismatch):", tg_path.name)
        continue

    sess_num, cond_str, cs_flag, rep = parsed
    session_id = sess_num
    cond_code = COND_CODE_MAP[cond_str]

    print(f"\nProcessing {tg_path.name} (session {session_id}, {cond_str})")

    # load mono audios
    path_A = mono_audio_path(sess_num, "A", cs_flag, rep)
    path_B = mono_audio_path(sess_num, "B", cs_flag, rep)

    snd_A = parselmouth.Sound(str(path_A))
    snd_B = parselmouth.Sound(str(path_B))

    sr_A = snd_A.sampling_frequency
    sr_B = snd_B.sampling_frequency

    if sr_A != sr_B:
        raise ValueError(f"Sampling rates differ in session {sess_num}: {sr_A} vs {sr_B}")

    # precompute Pitch for both speakers (for F0)
    pitch_A = snd_A.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=500)
    pitch_B = snd_B.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=500)


    tg = tgio.openTextgrid(str(tg_path))

    try:
        tier_diar_A   = tg.tierDict[TIER_DIAR_A]
        tier_diar_B   = tg.tierDict[TIER_DIAR_B]
        tier_events   = tg.tierDict[TIER_EVENTS]
        tier_syl_A    = tg.tierDict[TIER_SYL_A]
        tier_syl_B    = tg.tierDict[TIER_SYL_B]
    except KeyError as e:
        raise KeyError(
            f"Missing tier {e} in {tg_path.name}. "
            f"Available tiers: {list(tg.tierDict.keys())}"
        )



    # to process diarisation intervals for one speaker
    def process_diar_tier(diar_tier, speaker_id, snd, sr, pitch, syl_tier):
        for start, end, label in diar_tier.entryList:
            label = (label or "").strip()
            dur = end - start
            if dur <= 0:
                continue
            if label == "" or label.lower().startswith("silence"):
                continue

            low = label.lower()
            if speaker_id == 0 and "_a" not in low and low.endswith("a") is False:
                pass
            if speaker_id == 1 and "_b" not in low and low.endswith("b") is False:
                pass

            # exclude bc intervals
            if diar_interval_is_backchannel(tier_events, start, end, speaker_id):
                continue  # will be captured as bc via TransEvents

            # compute syllable-based speechrate
            n_syl = count_syllables_in_interval(syl_tier, start, end)
            speechrate = n_syl / dur if dur > 0 and n_syl > 0 else math.nan

            # F0 & RMS for this speaker
            F0 = mean_f0_in_interval(pitch, start, end)
            RMS = rms_in_interval(snd, sr, start, end)

            event_rows.append({
                "i": session_id,
                "c": cond_code,
                "speaker": speaker_id,      # 0=A, 1=B
                "event": "Turn",
                "duration": dur,
                "speechrate": speechrate,
                "F0": F0,
                "RMS": RMS,
                "time_sec": start,
            })

    # A's turns
    process_diar_tier(tier_diar_A, 0, snd_A, sr_A, pitch_A, tier_syl_A)
    # B's turns
    process_diar_tier(tier_diar_B, 1, snd_B, sr_B, pitch_B, tier_syl_B)

    #  for events: Gap, Silence, Overlap, bc

    for ev_start, ev_end, ev_label in tier_events.entryList:
        ev_label = (ev_label or "").strip()
        dur = ev_end - ev_start
        if dur <= 0 or ev_label == "":
            continue

        ev_type = classify_event_label(ev_label)
        if ev_type is None:
            continue  # ignore other labels like Turn_s##_A, Turn_s##_B etc.

        # bc have a speaker; others do not
        if ev_type == "Backchannel":
            # determine speaker
            low = ev_label.lower()
            if "_a" in low or low.endswith("a"):
                speaker_id = 0
                snd = snd_A
                sr = sr_A
                pitch = pitch_A
                syl_tier = tier_syl_A
            elif "_b" in low or low.endswith("b"):
                speaker_id = 1
                snd = snd_B
                sr = sr_B
                pitch = pitch_B
                syl_tier = tier_syl_B
            else:
                # ambiguous, skip
                continue

            n_syl = count_syllables_in_interval(syl_tier, ev_start, ev_end)
            speechrate = n_syl / dur if dur > 0 and n_syl > 0 else math.nan
            F0 = mean_f0_in_interval(pitch, ev_start, ev_end)
            RMS = rms_in_interval(snd, sr, ev_start, ev_end)
    
            event_rows.append({
                "i": session_id,
                "c": cond_code,
                "speaker": speaker_id,
                "event": "Backchannel",
                "duration": dur,
                "speechrate": speechrate,
                "F0": F0,
                "RMS": RMS,
                "time_sec": ev_start,
            })

        else:
            # Gap, Silence, Overlap – no single speaker
            event_rows.append({
                "i": session_id,
                "c": cond_code,
                "speaker": -1,       # no single speaker
                "event": ev_type,    # Gap/Silence/Overlap
                "duration": dur,
                "speechrate": math.nan,
                "F0": math.nan,
                "RMS": math.nan,
                "time_sec": ev_start,
            })

print("\nFinished collecting events.")


Found 30 combined TextGrids.

Processing ref_s05_NC1_processed.TextGrid (session 5, NC1)

Processing ref_s05_NC2_processed.TextGrid (session 5, NC2)

Processing ref_s06_NC1_processed.TextGrid (session 6, NC1)

Processing ref_s06_NC2_processed.TextGrid (session 6, NC2)

Processing ref_s07_NC1_processed.TextGrid (session 7, NC1)

Processing ref_s07_NC2_processed.TextGrid (session 7, NC2)

Processing ref_s10_NC1_processed.TextGrid (session 10, NC1)

Processing ref_s10_NC2_processed.TextGrid (session 10, NC2)

Processing ref_s11_NC1_processed.TextGrid (session 11, NC1)

Processing ref_s11_NC2_processed.TextGrid (session 11, NC2)

Processing ref_s12_NC1_processed.TextGrid (session 12, NC1)

Processing ref_s12_NC2_processed.TextGrid (session 12, NC2)

Processing ref_s13_NC1_processed.TextGrid (session 13, NC1)

Processing ref_s13_NC2_processed.TextGrid (session 13, NC2)

Processing ref_s16_NC1_processed.TextGrid (session 16, NC1)

Processing ref_s16_NC2_processed.TextGrid (session 16, NC2)



In [16]:
event_df = pd.DataFrame(event_rows)

# Optional: sort for readability
event_df = event_df.sort_values(["i", "c", "time_sec", "event"]).reset_index(drop=True)

save_path = ROOT / "descriptive_processed_event_data.csv"
event_df.to_csv(save_path, index=False)
print(f"Saved event data to {save_path}")

event_df.head(10)

Saved event data to /Users/moanason/Downloads/Data_ANA/descriptive_processed_event_data.csv


,i,c,speaker,event,duration,speechrate,F0,RMS,time_sec
0,5,0,1,Turn,11.356875,7.484453,256.353240,0.006155,1.988469
1,5,0,-1,Gap,0.371250,NaN,NaN,NaN,13.345344
2,5,0,0,Turn,9.095602,1.869035,109.146095,0.003943,13.716594
3,5,0,-1,Silence,1.011683,NaN,NaN,NaN,22.812195
4,5,0,0,Turn,2.005923,1.994095,121.429200,0.004715,23.823878
5,5,0,-1,Silence,0.710608,NaN,NaN,NaN,25.829801
6,5,0,0,Turn,2.965277,2.697893,104.359144,0.003429,26.540409
7,5,0,-1,Silence,1.343096,NaN,NaN,NaN,29.505686
8,5,0,0,Turn,3.927812,1.782163,93.160360,0.003293,30.848782
9,5,0,-1,Silence,1.198125,NaN,NaN,NaN,34.776594


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path("/Users/moanason/Downloads/Data_ANA")
EVENT_CSV = BASE_DIR / "descriptive_processed_event_data.csv"
OUT_CSV   = BASE_DIR / "descriptive_processed_event_data_with_pause.csv"

MIN_PAUSE = 0.0   # <10 ms, ignore smaller gaps, if needed
EPS = 1e-6        # tolerance

event_df = pd.read_csv(EVENT_CSV)

event_core = event_df.copy()

# sort by dyad, condition, and time
event_core = (
    event_core
    .sort_values(["i", "c", "time_sec", "speaker", "event"])
    .reset_index(drop=True)
)

print("Rows in event_core (without old Pause rows):", len(event_core))


def compute_pauses_for_group(group: pd.DataFrame):
    """
    group: all events for a given (i, c), already sorted by time_sec.

    Returns: list of dicts (new Pause events) for this dyad×condition.
    """
    pauses = []

    turns = (
        group[group["event"] == "Turn"]
        .sort_values("time_sec")
        .reset_index(drop=True)
    )

    if len(turns) < 2:
        return pauses

    for k in range(len(turns) - 1):
        cur = turns.iloc[k]
        nxt = turns.iloc[k + 1]

        # we only care about adjacent IPUs from the SAME speaker
        if cur["speaker"] != nxt["speaker"]:
            continue

        # compute gap between these two IPUs
        end_cur = cur["time_sec"] + cur["duration"]
        start_next = nxt["time_sec"]
        gap = start_next - end_cur

        if gap <= MIN_PAUSE + EPS:
            continue

        mask_sil = (
            (group["event"] == "Silence") &
            (group["time_sec"] < start_next - EPS) &
            ((group["time_sec"] + group["duration"]) > end_cur + EPS)
        )

        if not mask_sil.any():
            continue

        pauses.append(
            dict(
                i          = cur["i"],
                c          = cur["c"],
                speaker    = cur["speaker"],     # same speaker as the two IPUs
                event      = "Pause",
                duration   = float(gap),
                speechrate = np.nan,
                F0         = np.nan,
                RMS        = np.nan,
                time_sec   = float(end_cur),     # onset of the pause
                dyad       = cur.get("dyad", np.nan),
                condition  = cur.get("condition", np.nan),
            )
        )

    return pauses


# --------------------
all_pause_rows = []

for (i_val, c_val), grp in event_core.groupby(["i", "c"]):
    grp_sorted = grp.sort_values("time_sec").reset_index(drop=True)
    pauses_here = compute_pauses_for_group(grp_sorted)
    all_pause_rows.extend(pauses_here)

pause_df = pd.DataFrame(all_pause_rows)
print("Number of new Pause events:", len(pause_df))
print(pause_df.head())


# ensure same columns as event_core
for col in event_core.columns:
    if col not in pause_df.columns:
        pause_df[col] = np.nan

pause_df = pause_df[event_core.columns]

event_with_pause = (
    pd.concat([event_core, pause_df], ignore_index=True)
    .sort_values(["i", "c", "time_sec", "event"])
    .reset_index(drop=True)
)

event_with_pause.to_csv(OUT_CSV, index=False)
print("Saved updated event_df with Pause events to:", OUT_CSV)


Rows in event_core (without old Pause rows): 11023
Number of new Pause events: 1558
   i  c  speaker  event  duration  speechrate  F0  RMS   time_sec  dyad  \
0  5  0        0  Pause  1.011683         NaN NaN  NaN  22.812195   NaN   
1  5  0        0  Pause  0.710608         NaN NaN  NaN  25.829801   NaN   
2  5  0        0  Pause  1.343096         NaN NaN  NaN  29.505686   NaN   
3  5  0        0  Pause  1.198125         NaN NaN  NaN  34.776594   NaN   
4  5  0        0  Pause  0.436070         NaN NaN  NaN  41.400072   NaN   

   condition  
0        NaN  
1        NaN  
2        NaN  
3        NaN  
4        NaN  
Saved updated event_df with Pause events to: /Users/moanason/Downloads/Data_ANA/descriptive_processed_event_data_with_pause.csv


##### Demo analysis

In [18]:
# turn duration stats per speaker & condition
turn_stats = (
    event_df[event_df["event"] == "Turn"]
    .groupby(["c", "speaker"])["duration"]
    .agg(["count", "mean", "median"])
)
turn_stats

count      mean    median
c speaker                           
0 0         1019  3.991594  2.787737
  1         1119  3.850604  2.514375
1 0         1052  3.983307  2.667280
  1         1090  3.984487  2.765473

In [19]:
# gap and overlap duration distribution per condition
gap_stats = (
    event_df[event_df["event"] == "Gap"]
    .groupby("c")["duration"]
    .agg(["count", "mean", "median"])
)


overlap_stats = (
    event_df[event_df["event"] == "Overlap"]
    .groupby("c")["duration"]
    .agg(["count", "mean", "median"])
)

print(gap_stats)
print(overlap_stats)


   count      mean    median
c                           
0    552  0.553031  0.412924
1    478  0.586132  0.421875
   count      mean    median
c                           
0    549  0.872658  0.573750
1    692  0.787109  0.509405


In [ ]:
# bc rate per minute per condition
bc = event_df[event_df["event"] == "Backchannel"].copy()
conv_time = (
    event_df.groupby(["i", "c"])["duration"].sum()
    .rename("conv_dur")
    .reset_index()
)
bc_counts = (
    bc.groupby(["i", "c"]).size()
    .rename("n_bc")
    .reset_index()
)
bc_rate = bc_counts.merge(conv_time, on=["i", "c"])
bc_rate["bc_per_min"] = bc_rate["n_bc"] / (bc_rate["conv_dur"] / 60.0)
bc_rate.head()


,i,c,n_bc,conv_dur,bc_per_min
0,5,0,89,669.366386,7.977694
1,5,1,118,691.300401,10.241568
2,6,0,67,636.134960,6.319414
3,6,1,72,640.956264,6.739929
4,7,0,105,678.228361,9.288907
